# 01 - Análise Exploratória de Dados

**Exploração dos dados** · Dados Hospitalares - Diabetes 130-US Hospitals for Years 1999-2008

---

Este notebook percorre o **roteiro de oito etapas** da análise exploratória sobre a
base bruta de Diabetes 130-US Hospitals, usando o dicionário de dados montado em
[`00-dicionario-dados.ipynb`](00-dicionario-dados.ipynb) como mapa de tipos:

1. **Distribuição dos atributos** — o comportamento de cada característica analisada.
2. **Análise univariada** — distribuição de frequências e medidas de localização,
   dispersão, assimetria e curtose, uma variável por vez.
3. **Análise bivariada** — se a variação de uma característica é acompanhada pela
   variação de outra.
4. **Análise multivariada** — padrões e relações entre várias variáveis ao mesmo tempo.
5. **Valores ausentes** — onde falta dado e quanto essa falta pesa.
6. **Valores discrepantes** — registros cujo comportamento foge do esperado.
7. **Correlação** — qualquer relação estatística entre duas ou mais variáveis.
8. **Seleção de características** — a combinação de variáveis relevante para o
   aprendizado que vem depois.

Cada etapa grava as figuras em `docs/imagens/eda/` e a tabela que originou cada
figura em `reports/tables/eda-exploratoria/` — a regra do projeto é que
**nenhuma figura existe sem a tabela que a produziu**.


In [9]:
%matplotlib inline

def warn(*args, **kwargs):
    pass
# import warnings
# warnings.warn = warn
# warnings.simplefilter('ignore')

# Importações
import prince
import textwrap
import circlify
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.collections import EllipseCollection

In [17]:
# Funções para aplicar as escalas
def carregar_escalas(caminho):
    escala = pd.read_csv(caminho,
                         sep=None, engine='python',   # detecta , ou ;
                         dtype={'coluna': str, 'nivel': str},
                         encoding='utf-8-sig')        # remove BOM, se houver
    escala.columns = escala.columns.str.strip().str.lower()

    esperadas = {'coluna', 'nivel', 'ordem'}
    faltando = esperadas - set(escala.columns)
    if faltando:
        raise ValueError(f'Colunas ausentes em {caminho}: {faltando}. '
                         f'Colunas encontradas: {escala.columns.tolist()}')

    escala = escala.sort_values(['coluna', 'ordem'])
    return escala.groupby('coluna')['nivel'].apply(list).to_dict()


def aplicar_escalas(dados, escalas):
    for coluna, niveis in escalas.items():
        if coluna not in dados.columns:
            print(f'Aviso: a coluna "{coluna}" está na escala, mas não existe no dataset')
            continue

        # Se a coluna for numérica (ex.: notas 1 a 5), converte os níveis para número
        if pd.api.types.is_numeric_dtype(dados[coluna]):
            niveis = pd.to_numeric(pd.Series(niveis)).tolist()

        dados[coluna] = pd.Categorical(dados[coluna], categories=niveis, ordered=True)
    return dados

In [18]:
# Fazendo o mesmo processo de renomar as colunas do notebook 00
# Renomear todas as colunas (ingles para)
renomear_colunas = {
    'encounter_id': 'id_atendimento',
    'patient_nbr': 'id_paciente',
    'race': 'raca',
    'gender': 'sexo',
    'age': 'faixa_etaria',
    'weight': 'peso',
    'admission_type_id': 'id_tipo_admissao',
    'discharge_disposition_id': 'id_destino_alta',
    'admission_source_id': 'id_origem_admissao',
    'time_in_hospital': 'dias_internacao',
    'payer_code': 'codigo_fonte_pagadora',
    'medical_specialty': 'especialidade_medica',
    'num_lab_procedures': 'qtd_procedimentos_laboratoriais',
    'num_procedures': 'qtd_procedimentos',
    'num_medications': 'qtd_medicamentos',
    'number_outpatient': 'qtd_atendimentos_ambulatoriais',
    'number_emergency': 'qtd_atendimentos_emergenciais',
    'number_inpatient': 'qtd_internacoes_anteriores',
    'diag_1': 'diagnostico_principal',
    'diag_2': 'diagnostico_secundario_1',
    'diag_3': 'diagnostico_secundario_2',
    'number_diagnoses': 'qtd_diagnosticos',
    'max_glu_serum': 'resultado_maximo_glicemia',
    'A1Cresult': 'resultado_hemoglobina_glicada',
    'metformin': 'metformina',
    'repaglinide': 'repaglinida',
    'nateglinide': 'nateglinida',
    'chlorpropamide': 'clorpropamida',
    'glimepiride': 'glimepirida',
    'acetohexamide': 'acetohexamida',
    'glipizide': 'glipizida',
    'glyburide': 'gliburida',
    'tolbutamide': 'tolbutamida',
    'pioglitazone': 'pioglitazona',
    'rosiglitazone': 'rosiglitazona',
    'acarbose': 'acarbose',
    'miglitol': 'miglitol',
    'troglitazone': 'troglitazona',
    'tolazamide': 'tolazamida',
    'examide': 'examida',
    'citoglipton': 'citogliptona',
    'insulin': 'insulina',
    'glyburide-metformin': 'gliburida_metformina',
    'glipizide-metformin': 'glipizida_metformina',
    'glimepiride-pioglitazone': 'glimepirida_pioglitazona',
    'metformin-rosiglitazone': 'metformina_rosiglitazona',
    'metformin-pioglitazone': 'metformina_pioglitazona',
    'change': 'alteracao_medicacao',
    'diabetesMed': 'uso_medicacao_diabetes',
    'readmitted': 'situacao_reinternacao'
}

dados.rename(columns=renomear_colunas, inplace=True)

In [19]:
# Casos específicos da base adicional (admissão, alta e origem)
MAP_TIPO_ADMISSAO = {
    1: 'Emergency',
    2: 'Urgent',
    3: 'Elective',
    4: 'Newborn',
    5: 'Not Available',
    6: None,
    7: 'Trauma Center',
    8: 'Not Mapped'
}

MAP_DESTINO_ALTA = {
    1: 'Discharged to home',
    2: 'Transferred to short term hospital',
    3: 'Transferred to SNF',
    4: 'Transferred to ICF',
    5: 'Transferred to another inpatient care institution',
    6: 'Discharged to home with home health service',
    7: 'Left AMA',
    8: 'Discharged to home under Home IV provider',
    9: 'Admitted as inpatient to this hospital',
    10: 'Neonate transferred for neonatal aftercare',
    11: 'Expired',
    12: 'Still patient or expected outpatient return',
    13: 'Hospice / home',
    14: 'Hospice / medical facility',
    15: 'Transferred to Medicare approved swing bed',
    16: 'Referred to another institution for outpatient services',
    17: 'Referred to this institution for outpatient services',
    18: None,
    19: 'Expired at home',
    20: 'Expired in a medical facility',
    21: 'Expired, place unknown',
    22: 'Transferred to rehabilitation facility',
    23: 'Transferred to long term care hospital',
    24: 'Transferred to Medicaid nursing facility',
    25: 'Not Mapped',
    26: 'Unknown/Invalid',
    27: 'Transferred to federal health care facility',
    28: 'Transferred to psychiatric hospital',
    29: 'Transferred to Critical Access Hospital',
    30: 'Transferred to another health care institution'
}

MAP_ORIGEM_ADMISSAO = {
    1: 'Physician Referral',
    2: 'Clinic Referral',
    3: 'HMO Referral',
    4: 'Transfer from a hospital',
    5: 'Transfer from SNF',
    6: 'Transfer from another health care facility',
    7: 'Emergency Room',
    8: 'Court/Law Enforcement',
    9: 'Not Available',
    10: 'Transfer from critical access hospital',
    11: 'Normal Delivery',
    12: 'Premature Delivery',
    13: 'Sick Baby',
    14: 'Extramural Birth',
    15: 'Not Available',
    17: None,
    18: 'Transfer from another Home Health Agency',
    19: 'Readmission to same Home Health Agency',
    20: 'Not Mapped',
    21: 'Unknown/Invalid',
    22: 'Transfer from hospital inpatient/same facility',
    23: 'Born inside this hospital',
    24: 'Born outside this hospital',
    25: 'Transfer from Ambulatory Surgery Center',
    26: 'Transfer from Hospice'
}

In [20]:
MAPEAMENTOS = {
    'id_tipo_admissao': MAP_TIPO_ADMISSAO,
    'id_destino_alta': MAP_DESTINO_ALTA,
    'id_origem_admissao': MAP_ORIGEM_ADMISSAO
}

In [21]:
# Carregando base de dados
dados = pd.read_csv('../data/raw/diabetic_data.csv')

# Renomeando as colunas para português
dados.rename(columns=renomear_colunas, inplace=True)

# Carregando dicionário dos dados
dicionario = pd.read_csv('../data/processed/dicionario.csv', sep=';')

# Carregando escalas
escalas = carregar_escalas(
    caminho='../data/processed/escala.csv'
)

# Aplicando escalas
dados = aplicar_escalas(dados, escalas)

In [22]:
print(dados.shape)
print(dados.columns.tolist())

print('\nTipos das variáveis ordinais:')
print(dados[list(escalas.keys())].dtypes)

(101766, 50)
['id_atendimento', 'id_paciente', 'raca', 'sexo', 'faixa_etaria', 'peso', 'id_tipo_admissao', 'id_destino_alta', 'id_origem_admissao', 'dias_internacao', 'codigo_fonte_pagadora', 'especialidade_medica', 'qtd_procedimentos_laboratoriais', 'qtd_procedimentos', 'qtd_medicamentos', 'qtd_atendimentos_ambulatoriais', 'qtd_atendimentos_emergenciais', 'qtd_internacoes_anteriores', 'diagnostico_principal', 'diagnostico_secundario_1', 'diagnostico_secundario_2', 'qtd_diagnosticos', 'resultado_maximo_glicemia', 'resultado_hemoglobina_glicada', 'metformina', 'repaglinida', 'nateglinida', 'clorpropamida', 'glimepirida', 'acetohexamida', 'glipizida', 'gliburida', 'tolbutamida', 'pioglitazona', 'rosiglitazona', 'acarbose', 'miglitol', 'troglitazona', 'tolazamida', 'examida', 'citogliptona', 'insulina', 'gliburida_metformina', 'glipizida_metformina', 'glimepirida_pioglitazona', 'metformina_rosiglitazona', 'metformina_pioglitazona', 'alteracao_medicacao', 'uso_medicacao_diabetes', 'situaca

In [ ]:
# Mantive todas as 50 variaveis, apesar de 47 serem as possiveis de usar para um projeto não supervisionado.
# Motivo: podemos ter de usar alguma para decisao metodológica do projeto como pegar pacientes únicos e não atendimentos.
print('Colunas da base:', len(dados.columns))
print('Variáveis no dicionário:', len(dicionario))

# Variáveis da base que não aparecem no dicionário
print('\nNa base, mas não no dicionário:')
print(set(dados.columns) - set(dicionario['variavel']))

# Variáveis do dicionário que não aparecem na base
print('\nNo dicionário, mas não na base:')
print(set(dicionario['variavel']) - set(dados.columns))

Colunas da base: 50
Variáveis no dicionário: 50

Na base, mas não no dicionário:
set()

No dicionário, mas não na base:
set()


In [25]:
# selecionando as mesmas colunas do dicionário de dados
dados = dados.loc[:,dicionario.variavel]

In [26]:
dicionario

,variavel,descricao,valores,tipo,qtd_categorias,qtd_nan,qtd_interrogacao,mapeamento
0,id_atendimento,Identificador único do atendimento hospitalar,"[2278392, 149190, 64410, 500364, 16680, 35754,...",Identificador,101766,0,0,NaN
1,id_paciente,Identificador único do paciente,"[8222157, 55629189, 86047875, 82442376, 425192...",Identificador,71518,0,0,NaN
2,raca,Raça ou grupo étnico registrado para o paciente,"['Caucasian', 'AfricanAmerican', '?', 'Other',...",Nominal,6,0,2273,NaN
3,sexo,Sexo registrado do paciente,"['Female', 'Male', 'Unknown/Invalid']",Nominal,3,0,0,NaN
4,faixa_etaria,Faixa etária do paciente,"['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[...",Ordinal,10,0,0,NaN
5,peso,"Faixa de peso do paciente, em intervalos de 25...","['?', '[75-100)', '[50-75)', '[0-25)', '[100-1...",Ordinal,10,0,98569,NaN
6,id_tipo_admissao,Código que identifica o tipo de admissão hospi...,"[6, 1, 2, 3, 4, 5, 8, 7]",Nominal,8,0,0,"{1: 'Emergency', 2: 'Urgent', 3: 'Elective', 4..."
7,id_destino_alta,Código que identifica o destino do paciente ap...,"[25, 1, 3, 6, 2, 5, 11, 7, 10, 4, 14, 18, 8, 1...",Nominal,26,0,0,"{1: 'Discharged to home', 2: 'Transferred to s..."
8,id_origem_admissao,Código que identifica a origem da admissão hos...,"[1, 7, 2, 4, 5, 6, 20, 3, 17, 8, 9, 14, 10, 22...",Nominal,17,0,0,"{1: 'Physician Referral', 2: 'Clinic Referral'..."
9,dias_internacao,Quantidade de dias de permanência hospitalar,"[1, 3, 2, 4, 5, 13, 12, 9, 7, 10, 6, 11, 8, 14]",Discreta,14,0,0,NaN


## Análises iniciais

In [27]:
# Info geral
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                           Non-Null Count   Dtype   
---  ------                           --------------   -----   
 0   id_atendimento                   101766 non-null  int64   
 1   id_paciente                      101766 non-null  int64   
 2   raca                             101766 non-null  str     
 3   sexo                             101766 non-null  str     
 4   faixa_etaria                     101766 non-null  category
 5   peso                             3197 non-null    category
 6   id_tipo_admissao                 101766 non-null  int64   
 7   id_destino_alta                  101766 non-null  int64   
 8   id_origem_admissao               101766 non-null  int64   
 9   dias_internacao                  101766 non-null  int64   
 10  codigo_fonte_pagadora            101766 non-null  str     
 11  especialidade_medica             101766 non-null  str     
 12 

In [32]:
# Quantidade por tipo
dicionario.tipo.value_counts()

tipo
Nominal          36
Discreta          8
Ordinal           4
Identificador     2
Name: count, dtype: int64

- 36 variáveis nominais, incluindo diagnósticos, medicamentos e características demográficas.
- 8 variáveis discretas, principalmente contagens e quantidades.
- 4 variáveis ordinais, com hierarquia definida.
- 2 identificadores, que não devem entrar diretamente no algoritmo.

In [33]:
# Estatísticas descritivas
dados.describe()

,id_atendimento,id_paciente,id_tipo_admissao,id_destino_alta,id_origem_admissao,dias_internacao,qtd_procedimentos_laboratoriais,qtd_procedimentos,qtd_medicamentos,qtd_atendimentos_ambulatoriais,qtd_atendimentos_emergenciais,qtd_internacoes_anteriores,qtd_diagnosticos
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,1.026403e+08,3.869636e+07,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.252200e+04,1.350000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.523890e+08,4.550514e+07,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,2.302709e+08,8.754595e+07,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,4.438672e+08,1.895026e+08,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


## Análise univariada

### Características nominais

|Aspecto|O que investigar|Relevância para o projeto|
|---|---|---|
|Concentração|Uma categoria representa quase todos os registros?|Pode indicar variável pouco discriminativa.
|Desbalanceamento|Uma categoria é muito mais frequente que as demais?|Pode influenciar a formação dos grupos.
|Cardinalidade|Existem muitas categorias diferentes?|Pode exigir tratamento específico na codificação.
|Categorias desconhecidas|Aparecem ?, Unknown/Invalid ou Not Mapped?|Exige decisões de tratamento.
|Categorias raras|Existem categorias com pouquíssimos registros?|Podem gerar ruído ou grupos muito pequenos.

In [ ]:
# Obter uma lista das variáveis nominais
nominal_columns = (
    dicionario
    .query('tipo == "Nominal"')
    .variavel
    .to_list()
)

In [35]:
if nominal_columns != []:
    for column in dados.columns:
        if (column in nominal_columns):
            # Gerando gráfico
            ## Configurando parâmetros
            ax = sns.barplot(x=dados[column].value_counts().values[:10],y=dados[column].value_counts().index[:10], color='#183EFF',alpha=1)

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(25, 8)
            # Personalizando valores absolutos e relativos
            total = float(dados[column].value_counts().values.sum())
            for i in ax.containers:
                widths = [j.get_width() for j in i]
                total = sum(widths)
                ax.bar_label(i, label_type='edge', labels=[f"{width:.0f}\n{width/total:.2%}" for width in widths])

            ## Adicionando barra com valor médio
            ax.axvline(dados[column].value_counts().values.mean(), color="#FFD000", clip_on=False,ls='dashed')

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel('Quantidade de registros')
            plt.title('As categorias mais frequentes da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-nominal-'+column+'.png',bbox_inches='tight')
            plt.close(fig)

**Considerando a estrutura da nossa base, temos de dar atenção a estas:**
- **raca e sexo**: composição demográfica dos atendimentos.
- **especialidade_medica**: diversidade e concentração das especialidades.
- **diagnostico_principal, diagnostico_secundario_1 e diagnostico_secundario_2**: cardinalidade e concentração dos diagnósticos.
- **id_tipo_admissao, id_destino_alta e id_origem_admissao**: distribuição dos tipos de admissão, destinos de alta e origens.
- **metformina, insulina e demais medicamentos**: frequência de utilização e alterações na medicação.
- **situacao_reinternacao**: distribuição da variável que poderá ser utilizada posteriormente para caracterização dos agrupamentos, mas não como atributo do clustering.

In [ ]:
# Relatório para ver de forma vertical os dados de cada categotia
for column in nominal_columns:
    print(f"\n{'=' * 65}")
    print(f"VARIÁVEL: {column}")
    print(f"{'=' * 65}")

    frequencias = dados[column].value_counts(dropna=False)

    print(f"Categorias distintas: {dados[column].nunique()}")
    print(f"Registros válidos: {dados[column].notna().sum()}")
    print(f"Valores ausentes: {dados[column].isna().sum()}")

    print("\n10 categorias mais frequentes:")

    resumo = pd.DataFrame({
        'frequencia': frequencias.head(10),
        'percentual_total': (
            frequencias.head(10) / len(dados) * 100
        ).round(2)
    })

    print(resumo.to_string())


VARIÁVEL: raca
Categorias distintas: 6
Registros válidos: 101766
Valores ausentes: 0

10 categorias mais frequentes:
                 frequencia  percentual_total
raca                                         
Caucasian             76099             74.78
AfricanAmerican       19210             18.88
?                      2273              2.23
Hispanic               2037              2.00
Other                  1506              1.48
Asian                   641              0.63

VARIÁVEL: sexo
Categorias distintas: 3
Registros válidos: 101766
Valores ausentes: 0

10 categorias mais frequentes:
                 frequencia  percentual_total
sexo                                         
Female                54708             53.76
Male                  47055             46.24
Unknown/Invalid           3              0.00

VARIÁVEL: id_tipo_admissao
Categorias distintas: 8
Registros válidos: 101766
Valores ausentes: 0

10 categorias mais frequentes:
                  frequencia  percen

In [42]:
# Identificando ocorrências do marcador "?"
for coluna in dados.columns:
    quantidade = (dados[coluna] == '?').sum()

    if quantidade > 0:
        percentual = quantidade / len(dados) * 100

        print(
            f'{coluna}: {quantidade:,} '
            f'({percentual:.2f}%)'
        )

raca: 2,273 (2.23%)
codigo_fonte_pagadora: 40,256 (39.56%)
especialidade_medica: 49,949 (49.08%)
diagnostico_principal: 21 (0.02%)
diagnostico_secundario_1: 358 (0.35%)
diagnostico_secundario_2: 1,423 (1.40%)


- Especialidade médica: não há informação sobre a especialidade associada ao atendimento. Isso não significa necessariamente que o atendimento não tenha sido realizado por um especialista.
- Fonte pagadora: não há informação identificada sobre a fonte pagadora.
- Raça: a informação não foi registrada ou disponibilizada.

> Acho interessante ainda não tratar isso. Devemos continuar a análise e na etapa de pré-processamento, tomarmos uma decisão sobre o que realizar com esses valores que não estão ausentes mas estão inespecíficaos.

## Análise bivariada